# Search Keyword Performance Analysis

Visualises the gold-layer output from the Adobe Analytics hit-level pipeline.

**Data source:** `output/YYYY-mm-dd_SearchKeywordPerformance.tab`  
**Business question:** Which external search engines and keywords drive the most revenue?

In [ ]:
import glob
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Load gold output (latest file in output/) ──────────────────────────────
OUTPUT_DIR = os.path.join(os.path.dirname(os.getcwd()), 'output') if 'notebooks' in os.getcwd() else 'output'
files = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*_SearchKeywordPerformance.tab')))

if not files:
    # Fallback: run the analyzer on the sample data right now
    sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src') if 'notebooks' in os.getcwd() else 'src')
    from shared.search_keyword_analyzer import SearchKeywordAnalyzer
    data_file = os.path.join(os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else '.', 'data', 'data.sql')
    analyzer = SearchKeywordAnalyzer(data_file)
    analyzer.process()
    output_path = analyzer.write_output(OUTPUT_DIR)
    files = [output_path]
    print(f'Generated: {output_path}')

latest = files[-1]
print(f'Loading: {latest}')
df = pd.read_csv(latest, sep='\t')
df['Label'] = df['Search Engine Domain'] + '\n"' + df['Search Keyword'] + '"'
print(df.to_string(index=False))

## Chart 1 — Revenue by Keyword (Bar)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#e34c26' if 'google' in d else '#00a4ef' if 'bing' in d else '#7b0099'
          for d in df['Search Engine Domain']]

bars = ax.barh(df['Label'][::-1], df['Revenue'][::-1], color=colors[::-1], edgecolor='white', height=0.55)

for bar, rev in zip(bars, df['Revenue'][::-1]):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height() / 2,
            f'${rev:,.2f}', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Revenue ($)', fontsize=12)
ax.set_title('Search Keyword Performance — Revenue by Keyword', fontsize=14, fontweight='bold', pad=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xlim(0, df['Revenue'].max() * 1.25)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', linestyle='--', alpha=0.4)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e34c26', label='Google'),
    Patch(facecolor='#00a4ef', label='Bing'),
    Patch(facecolor='#7b0099', label='Yahoo'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'chart_revenue_by_keyword.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart_revenue_by_keyword.png')

## Chart 2 — Revenue Share by Search Engine (Pie)

In [ ]:
engine_revenue = df.groupby('Search Engine Domain')['Revenue'].sum().sort_values(ascending=False)

engine_colors = {'google.com': '#e34c26', 'bing.com': '#00a4ef', 'search.yahoo.com': '#7b0099'}
pie_colors = [engine_colors.get(e, '#999999') for e in engine_revenue.index]

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    engine_revenue,
    labels=engine_revenue.index,
    autopct=lambda p: f'{p:.1f}%\n(${p/100*engine_revenue.sum():,.2f})',
    colors=pie_colors,
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12},
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')

ax.set_title(f'Revenue Share by Search Engine\nTotal: ${engine_revenue.sum():,.2f}',
             fontsize=14, fontweight='bold', pad=16)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'chart_revenue_by_engine.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart_revenue_by_engine.png')

## Chart 3 — Keyword Contribution Within Each Engine (Stacked)

In [ ]:
pivot = df.pivot_table(index='Search Engine Domain', columns='Search Keyword', values='Revenue', aggfunc='sum').fillna(0)

fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot(kind='bar', ax=ax, edgecolor='white', width=0.5)

ax.set_xlabel('Search Engine', fontsize=12)
ax.set_ylabel('Revenue ($)', fontsize=12)
ax.set_title('Revenue by Engine and Keyword', fontsize=14, fontweight='bold', pad=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.legend(title='Keyword', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'chart_keyword_by_engine.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart_keyword_by_engine.png')

## Summary Table

In [ ]:
summary = df[['Search Engine Domain', 'Search Keyword', 'Revenue']].copy()
summary['Revenue'] = summary['Revenue'].apply(lambda x: f'${x:,.2f}')
summary['% of Total'] = (df['Revenue'] / df['Revenue'].sum() * 100).apply(lambda x: f'{x:.1f}%')
summary.index = range(1, len(summary) + 1)
summary